# Exercice 1 - Découvrez les blocs de construction de l'Apprentissage par Renforcement

## 🎯 Objectif pédagogique

Dans cet exercice, nous allons nous familiariser avec les **3 blocs fondamentaux** de l'apprentissage par renforcement :

```
  ┌──────────────┐
  │ Observation  │ (ce que l'agent "voit")
  └──────┬───────┘
         │
         ▼
  ┌──────────────┐
  │    Action    │ (ce que l'agent "décide")
  └──────┬───────┘
         │
         ▼
  ┌──────────────┐
  │  Récompense  │ (feedback de l'environnement)
  └──────────────┘
```

Ce cycle observation → action → récompense est le **cœur du RL**.

## 🕹️ Contexte : Pourquoi CartPole ?

**CartPole-v1** est un environnement classique de RL car :
- Il est **simple** : un chariot + un pendule
- Il est **borné** : on sait quand on échoue
- Il est **rapide** : un épisode dure quelques secondes
- Il illustre parfaitement le cycle fondamental du RL

**Règle du jeu :**
- Le chariot peut aller à gauche ou à droite
- Le pendule doit rester debout (angle < 15°)
- L'épisode se termine si le pendule tombe ou après 500 steps

**Objectif :** Tenir le plus longtemps possible pour accumuler des récompenses

## 1. Import des bibliothèques

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

**Explication :**
- `gymnasium` : la bibliothèque qui fournit les environnements de RL
- `numpy` : pour les calculs mathématiques
- `matplotlib` : pour visualiser les résultats

## 2. Partie 1 : Exploration de l'environnement

Avant de faire apprendre un agent, nous devons **comprendre le terrain de jeu**.

### Création de l'environnement

In [ ]:
# Créer l'environnement CartPole-v1
env = gym.make("CartPole-v1")

print(f"Environnement créé : {env}")
print(f"Horizon (max steps par épisode) : {env.spec.max_episode_steps}")

**💡 Explication :**

`gym.make()` crée une instance de l'environnement. Chaque environnement a :
- Un **espace d'observation** (ce que l'agent perçoit)
- Un **espace d'action** (ce que l'agent peut faire)

`max_episode_steps = 500` signifie qu'un épisode dure maximum 500 steps, soit environ 25 secondes à 20 steps/seconde.

### L'espace d'observation

In [ ]:
print("=== Espace d'observation ===")
print(f"Type : {env.observation_space}")
print(f"Type de données : {env.observation_space.dtype}")
print(f"\nBornes basses : {env.observation_space.low}")
print(f"Bornes hautes : {env.observation_space.high}")

# Exemple d'observation aléatoire
exemple = env.observation_space.sample()
print(f"\nExemple d'observation : {exemple}")

**💡 Explication détaillée :**

L'agent reçoit **4 valeurs continues** à chaque step :

| Index | Nom | Description | Bornes |
|-------|-----|-------------|--------|
| 0 | `x` | Position horizontale du chariot | -4.8 à 4.8 |
| 1 | `x_dot` | Vitesse du chariot | illimité |
| 2 | `theta` | Angle du pendule (rad) | -0.42 à 0.42 (~24°) |
| 3 | `theta_dot` | Vitesse angulaire | illimité |

**Question pour le mentor :** Pourquoi certaines valeurs sont "illimitées" (inf) ?

→ Parce que la physique permet des vitesses infinies en théorie. En pratique, elles restent bornées par les lois physiques.

### L'espace d'action

In [ ]:
print("=== Espace d'action ===")
print(f"Type : {env.action_space}")
print(f"Nombre d'actions possibles : {env.action_space.n}")

# Exemple d'actions aléatoires
actions = [env.action_space.sample() for _ in range(5)]
print(f"\nExemples d'actions : {actions}")

**💡 Explication :**

`Discrete(2)` signifie qu'il y a **2 actions possibles** :

| Action | Effet |
|--------|-------|
| 0 | Pousser le chariot à **gauche** |
| 1 | Pousser le chariot à **droite** |

**Question pour le mentor :** Pourquoi y a-t-il seulement 2 actions ?

→ Car CartPole est un problème à **actions discrètes**. D'autres environnements (comme LunarLander) ont des actions continues (puissance du moteur, angle, etc.).

### 📝 Synthèse de la Partie 1

**Ce qu'on a appris :**
- ✅ CartPole-v1 a un **espace d'observation continu** (Box) de 4 dimensions
- ✅ CartPole-v1 a un **espace d'action discret** (Discrete) de 2 actions
- ✅ L'agent "voit" la position, vitesse, angle et vitesse angulaire
- ✅ L'agent "décide" d'aller à gauche ou à droite

**Concepts clés :**
- **Observation space** = ce que l'agent perçoit (input du modèle)
- **Action space** = ce que l'agent peut faire (output du modèle)
- **Box vs Discrete** = continu vs discret, fondamental pour choisir l'algorithme

## 3. Partie 2 : La boucle d'interaction agent-environnement

Maintenant que nous comprenons les espaces, voyons comment l'agent **interagit** avec l'environnement.

### Le cycle fondamental : reset() et step()

In [ ]:
# Réinitialiser l'environnement - C'EST OBLIGATOIRE avant le premier step
observation, info = env.reset()
print(f"État initial (observation) : {observation}")
print(f"Info supplémentaire : {info}")

**💡 Explication de reset() :**

`reset()` fait 2 choses :
1. **Réinitialise** l'état du système (chariot au centre, pendule vertical)
2. **Retourne** l'observation initiale

`reset()` doit être appelé **au début de chaque épisode**.

### Exécuter UN step

In [ ]:
# Choisir une action
action = env.action_space.sample()  # Aléatoire pour l'instant
print(f"Action choisie : {action} (0=gauche, 1=droite)")

# Exécuter l'action et observer le résultat
observation, reward, terminated, truncated, info = env.step(action)

print(f"\n=== Résultat de step() ===")
print(f"Nouvelle observation : {observation}")
print(f"Récompense (reward) : {reward}")
print(f"Terminated : {terminated} (épisode terminé ?)")
print(f"Truncated : {truncated} (coupure ?)")
print(f"Info : {info}")

**💡 Explication détaillée de step() :**

La fonction `step(action)` retourne **5 valeurs** :

| Variable | Type | Signification |
|----------|------|---------------|
| `observation` | array(4,) | Nouvel état du système |
| `reward` | float | Récompense reçue (ici = 1 si survive, 0 si mort) |
| `terminated` | bool | TRUE si l'épisode est terminé naturellement (pendule tombé) |
| `truncated` | bool | TRUE si l'épisode est coupé (temps écoulé, 500 steps) |
| `info` | dict | Infos supplémentaires (debug, métriques) |

**Question pour le mentor :** Pourquoi 2 variables pour "fin d'épisode" ?

→ `terminated` = fin "naturelle" (succès ou échec)
→ `truncated` = fin "artificielle" (limite de temps/steps)

L'épisode est **terminé** si `terminated=True` OU `truncated=True`.

### Exécuter un ÉPISODE complet

In [ ]:
# Exécuter un épisode complet avec un agent ALÉATOIRE
total_reward = 0
steps = 0
terminated = truncated = False

while not (terminated or truncated):
    # 1. Choisir une action
    action = env.action_space.sample()  # Aléatoire
    
    # 2. L'exécuter
    observation, reward, terminated, truncated, info = env.step(action)
    
    # 3. Accumuler la récompense
    total_reward += reward
    steps += 1

print(f"=== Épisode terminé ===")
print(f"Durée : {steps} steps")
print(f"Reward total : {total_reward}")
print(f"Raison : {'terminated' if terminated else 'truncated'}")

**💡 Explication de la boucle :**

La boucle `while` continue jusqu'à ce que l'épisode se termine.

**Question pour le mentor :** Pourquoi `reward` est presque toujours = 1 ?

→ Dans CartPole, la récompense est **par step de survie** :
- +1 point à chaque step où le pendule ne tombe pas
- 0 point quand le pendule tombe

L'objectif est donc de **survivre le plus longtemps possible** pour accumuler des points.

### Boucle sur 10 épisodes

In [ ]:
# Répéter l'expérience sur 10 épisodes
n_episodes = 10
rewards_per_episode = []

for episode in range(n_episodes):
    # Réinitialiser l'environnement pour ce nouvel épisode
    observation, info = env.reset()
    
    # Variables pour cet épisode
    total_reward = 0
    terminated = truncated = False
    
    # Boucle de l'épisode
    while not (terminated or truncated):
        action = env.action_space.sample()  # Aléatoire
        observation, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
    
    # Stocker le résultat
    rewards_per_episode.append(total_reward)
    print(f"Épisode {episode+1:2d}: {total_reward:3.0f} points")

# Fermer l'environnement (toujours faire ça à la fin !)
env.close()

**💡 Points importants à retenir :**

1. **reset() au début de chaque épisode** - sinon l'environnement garde l'état précédent
2. **Réinitialiser les variables** (`total_reward`, `terminated`, `truncated`) pour chaque épisode
3. **env.close() à la fin** - libère les ressources

**Question pour le mentor :** Pourquoi les scores varient-ils autant (10 à 32) ?

→ L'agent est **aléatoire** : il a 50% de chances de choisir la bonne action à chaque step. C'est le **hasard** qui détermine la durée de chaque épisode.

### 📝 Synthèse de la Partie 2

**Ce qu'on a appris :**
- ✅ Le cycle fondamental : `reset()` → `step()` en boucle → `close()`
- ✅ `step()` retourne 5 valeurs : observation, reward, terminated, truncated, info
- ✅ Un épisode = une séquence de steps jusqu'à `terminated=True` ou `truncated=True`
- ✅ L'agent aléatoire fait ~10-50 points (score faible)

**Concepts clés :**
- **Episode** = une tentative complète (reset à close)
- **Step** = une action + son résultat
- **Reward** = feedback de l'environnement (guide l'apprentissage)

## 4. Partie 3 : Visualisation des résultats

In [ ]:
# Visualiser la performance de l'agent aléatoire
plt.figure(figsize=(10, 4))
plt.bar(range(1, n_episodes + 1), rewards_per_episode, color='steelblue', alpha=0.7)
plt.axhline(y=np.mean(rewards_per_episode), color='red', linestyle='--', linewidth=2,
            label=f'Moyenne: {np.mean(rewards_per_episode):.1f} points')
plt.xlabel('Épisode')
plt.ylabel('Reward total')
plt.title('Performance de l\'agent aléatoire sur CartPole-v1')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.ylim(0, 550)  # Le maximum théorique est 500
plt.show()

### Statistiques

In [ ]:
print(f"=== Statistiques de l'agent aléatoire ===")
print(f"Reward moyen   : {np.mean(rewards_per_episode):.1f}")
print(f"Reward max     : {np.max(rewards_per_episode)}")
print(f"Reward min     : {np.min(rewards_per_episode)}")
print(f"Écart-type     : {np.std(rewards_per_episode):.1f}")
print(f"\nScore théorique maximum : 500 points (500 steps)")

## 5. Conclusion et Questions pour le Mentor

### Ce qu'on a accompli

| Compétence | Status |
|-----------|--------|
| Comprendre l'espace d'observation | ✅ |
| Comprendre l'espace d'action | ✅ |
| Implémenter la boucle reset/step/close | ✅ |
| Interpréter les 5 valeurs de step() | ✅ |
| Analyser les résultats | ✅ |

### Questions à discuter avec le mentor

1. **Pourquoi** CartPole est-il un bon premier environnement pour le RL ?
2. **Quelle serait** la différence avec un environnement à actions continues ?
3. **Comment** pourrait-on améliorer l'agent (sans encore parler d'apprentissage) ?
4. **Quel** est le lien entre le score de 17 et l'objectif de 500 ?

### Observations clés

- L'agent **aléatoire** obtient en moyenne **~20 points** (faible)
- Le score maximum théorique est **500 points** (25 secondes)
- Il y a donc une **énorme marge d'amélioration** !

### Prochaines étapes

- **Exercice 2 :** Implémenter un Q-Learning avec une table de décision
- **Exercice 3 :** Passer à un réseau de neurones (DQN)
- **Mission :** Appliquer au LunarLander (score > 200)